# FIFA World Cup — Modeling & 2026 Predictions

Covers **Steps 3–7**: target definition, the leakage-free feature set, baseline
and advanced models, time-based validation, and the 2026 forecast.

All heavy lifting lives in the reusable `src/` package; this notebook is the
narrated walk-through. The same logic runs headless via
`python scripts/run_pipeline.py`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import (config, data, evaluate, features, poisson,  # noqa: E402
                 predict, train)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIG = config.FIGURES_DIR

## 3. Target definition & 4. The modeling dataset

**Target** (3-class, from `team_a`'s perspective): `team_a_win` / `draw` /
`team_b_win`, derived from the **regulation+ET goal score** (penalty shoot-outs
count as draws — see EDA).

**Neutral teams.** Because venues are neutral and the raw home/away label is
biased, each match is emitted **once** with `team_a`/`team_b` assigned by a
seeded coin flip, and the model consumes **antisymmetric difference features**
(`team_a − team_b`). Swapping the two teams flips every feature's sign and flips
the label — exactly the symmetry a neutral-venue model should respect.

**Leakage prevention.** Every feature is computed in a single chronological pass
that records each team's state *before* applying the match result, so a match's
features only ever see strictly-earlier matches.

In [2]:
matches = data.load_men_matches()
df = features.build_modeling_dataset(matches)

print("Modeling dataset:", df.shape)
print("\nTarget distribution (balanced after symmetrisation):")
print(df["target"].map(config.INT_TO_OUTCOME).value_counts(normalize=True).round(3))
print("\nFeatures used:")
print(features.FEATURE_COLUMNS)

df[["year", "team_a_name", "team_b_name", "elo_diff", "form_points_diff",
    "host_advantage", "h2h_played", "goals_a", "goals_b", "target"]].head()

Modeling dataset: (960, 34)

Target distribution (balanced after symmetrisation):
target
team_b_win    0.394
team_a_win    0.388
draw          0.219
Name: proportion, dtype: float64

Features used:
['elo_diff', 'matches_played_diff', 'win_rate_diff', 'gf_avg_diff', 'ga_avg_diff', 'form_points_diff', 'experience_diff', 'host_advantage', 'h2h_played', 'h2h_a_win_rate', 'h2h_goal_diff_avg', 'same_confederation', 'knockout_stage']


,year,team_a_name,team_b_name,elo_diff,form_points_diff,host_advantage,h2h_played,goals_a,goals_b,target
0,1930,France,Mexico,0.0,NaN,0,0,4,1,0
1,1930,Belgium,United States,0.0,NaN,0,0,0,3,2
2,1930,Yugoslavia,Brazil,0.0,NaN,0,0,2,1,0
3,1930,Romania,Peru,0.0,NaN,0,0,3,1,0
4,1930,France,Argentina,35.0,NaN,0,0,0,1,2


## 5 & 6. Models and time-based validation

**Why not a random split?** Features (Elo, cumulative stats) are built from the
past, and a random split would leak future tournaments through the shared
rating state. We instead run a **temporal backtest**: to predict tournament `Y`,
train only on matches from years `< Y`. We evaluate the last four World Cups
(2010–2022).

**Metrics.** Accuracy is intuitive but misleading (draws are rarely the modal
class). The honest, *primary* metric is **log loss** — a proper scoring rule
that rewards calibrated probabilities — complemented by the **Brier score**.
Macro-F1 checks the draw class isn't ignored.

In [3]:
comparison = evaluate.compare_models(df)
comparison

,model,n,accuracy,macro_f1,log_loss,brier
0,random_forest,256,0.554688,0.416931,1.005776,0.597712
1,logistic_regression,256,0.496094,0.388929,1.040500,0.613913
2,xgboost,256,0.476562,0.423672,1.115755,0.655918
3,lightgbm,256,0.453125,0.399911,1.135210,0.666088
4,hist_gradient_boosting,256,0.457031,0.412149,1.217051,0.694897
5,prior_baseline,256,0.398438,0.356611,21.682510,1.203125
6,majority_baseline,256,0.382812,0.184557,22.245692,1.234375


The two **baselines** (majority class, prior probabilities) have catastrophic
log loss — they are uncalibrated. **Logistic regression** and **Random Forest**
both beat them comfortably; the gradient-boosting models do not improve on a
small (~960-match) dataset. We pick the best by log loss (Random Forest) and
inspect it per tournament.

In [4]:
best = comparison[~comparison["model"].str.contains("baseline")].iloc[0]["model"]
print("Best model by log loss:", best)

per_year, overall = evaluate.temporal_backtest(
    df, lambda: train.build_models()[best])
display(per_year)

# pooled confusion matrix across the backtest years
pt, pp = [], []
for ty in config.BACKTEST_YEARS:
    tr, te = df[df["year"] < ty], df[df["year"] == ty]
    if len(tr) and len(te):
        mdl = train.fit_model(train.build_models()[best], tr)
        pp.append(train.predict_proba(mdl, te)); pt.append(te["target"].to_numpy())
cm = evaluate.confusion(np.concatenate(pt), np.concatenate(pp))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Confusion matrix — {best} (pooled backtest)")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.show()

Best model by log loss: random_forest


,year,accuracy,macro_f1,log_loss,brier,n
0,2010,0.515625,0.396104,1.037819,0.624403,64
1,2014,0.562500,0.417527,0.997119,0.592759,64
2,2018,0.578125,0.429293,0.993896,0.589447,64
3,2022,0.562500,0.425078,0.994267,0.584239,64


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_10497/2969281482.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "confusion_matrix.png", dpi=110); plt.show()


The confusion matrix shows the expected behaviour: the model **almost never
predicts a draw as the most-likely class** (draws are genuinely hard and never
the modal outcome of an individual neutral match). It still assigns ~20–30%
*probability* to draws, which is why probabilistic metrics are the right lens.

### Feature importance / coefficients

In [5]:
# For Random Forest, show the Gini importances
rf = train.fit_model(train.build_models()[best], df)
if hasattr(rf, "feature_importances_"):
    imp = pd.Series(rf.feature_importances_, index=features.FEATURE_COLUMNS).sort_values()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    imp.plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_title(f"Feature importances — {best}")
    plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=110); plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_10497/2919967150.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=110); plt.show()


### Poisson goal model

We also fit a **Poisson regression** that predicts each team's expected goals
as a function of its attacking rate, the opponent's defensive rate, the Elo gap,
and whether it's a knockout match. This gives exact score predictions.

In [6]:
pois = poisson.PoissonGoalModel().fit(df)

# Poisson-derived outcome probabilities for the backtest
pois_pt, pois_pp = [], []
for ty in config.BACKTEST_YEARS:
    tr, te = df[df["year"] < ty], df[df["year"] == ty]
    if len(tr) and len(te):
        pm = poisson.PoissonGoalModel().fit(tr)
        pois_pp.append(pm.predict_proba(te)); pois_pt.append(te["target"].to_numpy())
pois_metrics = evaluate.compute_metrics(np.concatenate(pois_pt),
                                        np.concatenate(pois_pp))
print("Poisson model (pooled backtest):", pois_metrics)

Poisson model (pooled backtest): {'accuracy': 0.52734375, 'macro_f1': 0.3958333333333333, 'log_loss': 1.0131676021352998, 'brier': 0.6035847787561529, 'n': 256}


## 7. FIFA World Cup 2026 — Predictions

We train the best model and the Poisson model on **all** available data
(1930–2022), snapshot the final Elo/stats state, and predict the 72
group-stage matches from the official-draw-based fixture. We also run a
Monte-Carlo tournament simulation to estimate each team's chance of lifting the
trophy.

In [7]:
from scripts.build_fixture_2026 import GROUPS, HOSTS  # noqa: E402

clf, pois_model, hist, clf_name = predict.fit_full(best)
print(f"Using classifier: {clf_name}")

fixture = pd.read_csv(config.FIXTURE_2026_PATH)
_, unmatched = predict.resolve_team_ids(
    sorted(set(fixture["team_a"]) | set(fixture["team_b"])))
if unmatched:
    print(f"Debutant teams (no men's WC history, Elo={int(config.ELO_BASE)}):",
          unmatched)

preds = predict.predict_matches(fixture, clf, pois_model, hist, HOSTS)
cols = ["match_number", "group", "team_a", "team_b", "team_a_elo",
        "team_b_elo", "p_team_a_win", "p_draw", "p_team_b_win",
        "predicted_outcome_label", "predicted_score",
        "exp_goals_a", "exp_goals_b"]
preds[cols].to_csv(config.PROCESSED_DIR / "predictions_2026_groups.csv",
                   index=False)
preds[cols]

Using classifier: random_forest
Debutant teams (no men's WC history, Elo=1500): ['Cape Verde', 'Curacao', 'Jordan', 'Uzbekistan']


,match_number,group,team_a,team_b,team_a_elo,team_b_elo,p_team_a_win,p_draw,p_team_b_win,predicted_outcome_label,predicted_score,exp_goals_a,exp_goals_b
0,1,A,Mexico,South Africa,1530,1483,0.5440,0.2140,0.2420,Mexico,1-1,1.26,1.16
1,2,A,South Korea,Czech Republic,1431,1477,0.4088,0.2479,0.3433,South Korea,1-1,1.07,1.30
2,3,B,Canada,Bosnia and Herzegovina,1397,1491,0.1924,0.2035,0.6042,Bosnia and Herzegovina,0-1,0.89,1.45
3,4,D,United States,Paraguay,1445,1516,0.4188,0.2577,0.3235,United States,1-1,1.06,1.32
4,5,B,Qatar,Switzerland,1439,1498,0.1168,0.1910,0.6923,Switzerland,0-1,0.98,1.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,1610,1479,0.5541,0.2195,0.2264,Croatia,1-0,1.44,0.97
68,69,K,Colombia,Portugal,1578,1599,0.2579,0.2947,0.4474,Portugal,1-1,1.16,1.32
69,70,K,DR Congo,Uzbekistan,1417,1500,0.3418,0.1754,0.4827,Uzbekistan,0-2,0.87,2.18
70,71,J,Algeria,Austria,1449,1510,0.2222,0.3116,0.4662,Austria,1-1,1.09,1.33


### Monte-Carlo tournament simulation

In [8]:
sim = predict.simulate_tournament(GROUPS, clf, pois_model, hist, HOSTS,
                                  n_sims=3000)
sim.to_csv(config.PROCESSED_DIR / "tournament_simulation_2026.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_n = sim.head(20)
bars = ax.barh(top_n["team"][::-1], top_n["p_champion"][::-1], color="#4C72B0")
ax.set_xlabel("P(Champion)")
ax.set_title("Top 20 title contenders — FIFA World Cup 2026")
for b in bars:
    ax.text(b.get_width() + 0.002, b.get_y() + b.get_height() / 2,
            f"{b.get_width():.1%}", va="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "title_probabilities.png", dpi=110); plt.show()

sim.head(20)

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_10497/935055390.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "title_probabilities.png", dpi=110); plt.show()


,team,elo,p_advance,p_champion
0,Brazil,1724,0.9550,0.2013
1,France,1775,0.9733,0.1267
2,Argentina,1713,0.9420,0.1217
3,Germany,1741,0.9037,0.0850
4,Netherlands,1804,0.8930,0.0800
5,England,1640,0.9377,0.0610
6,Spain,1631,0.8797,0.0573
7,Portugal,1598,0.9257,0.0377
8,Belgium,1625,0.9470,0.0267
9,Sweden,1596,0.8577,0.0250


## Interpretation & caveats

These predictions are only as good as the *historical World Cup data* allows:

* **No data between tournaments.** Elo ratings only update every 4 years at the
  World Cup — real team strength changes in between (qualifiers, friendlies,
  continental cups). External Elo/FIFA ranking data would substantially help.
* **Squad and player changes** are invisible to a results-only model. 2022
  champions Argentina may have a weaker/stronger squad in 2026; the model cannot
  know this.
* **Defunct teams.** West Germany, USSR, Yugoslavia hold high Elo that modern
  Germany, Russia, Serbia do not inherit. Germany's current rating is based only
  on matches played under that exact name in the dataset.
* **48-team expansion.** 2026 has a new format (12 groups of 4, 32 R-of-32).
  This is structurally different from anything the model trained on; the
  tournament-simulation bracket rules are an approximation.
* **Draw class remains hard.** No model reliably predicts when a match will be a
  draw; probabilities ~20–25% are the best that pure-results models achieve.

For a production forecaster, the next-highest-value improvements would be:
1. Incorporate international matches outside the World Cup (doubles the data).
2. Use the official FIFA/Elo ranking (much finer-grained strength estimates).
3. Add squad-level data (injuries, market value, age profile).
4. Use betting-market implied probabilities as calibration targets.